In [0]:
# Databricks notebook source

# COMMAND ----------
# MAGIC %md
# MAGIC # EIOPA RFR Curve Analysis — Article #10
# MAGIC EUR Spot Rate / Forward Rate / Discount Factor / BEL Impact

# COMMAND ----------
# Cell 1: Setup
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams['font.size'] = 11

# ── Constants ──
LLP = 20          # Last Liquid Point (EUR)
UFR = 0.0330      # Ultimate Forward Rate (EUR, 2027)
CONV = 60         # Convergence Point

# ── Colors & Labels ──
COLORS = {
    "RFR_20251231_noVA":   "#1a5276",
    "RFR_20251231_withVA": "#2e86c1",
    "RFR_20260331_noVA":   "#922b21",
    "RFR_20260331_withVA": "#e74c3c",
}
LABELS = {
    "RFR_20251231_noVA":   "2025-Q4 no VA",
    "RFR_20251231_withVA": "2025-Q4 with VA",
    "RFR_20260331_noVA":   "2026-Q1 no VA",
    "RFR_20260331_withVA": "2026-Q1 with VA",
}

# COMMAND ----------
# Cell 2: Load data from discount_curve table

df_raw = spark.sql("""
    select version_id, tenor_month, zero_rate_annual
    from workspace.life_insurance_raw.discount_curve
    order by version_id, tenor_month
""").toPandas()

df_raw["tenor_year"] = df_raw["tenor_month"] / 12

# Build dict structure
curves = {}
for vid in df_raw["version_id"].unique():
    sub = df_raw[df_raw["version_id"] == vid].sort_values("tenor_year")
    curves[vid] = {
        "tenors": sub["tenor_year"].astype(int).tolist(),
        "spots":  sub["zero_rate_annual"].astype(float).tolist(),
    }

print(f"Loaded {len(curves)} curves: {list(curves.keys())}")
for vid, c in curves.items():
    print(f"  {vid}: {len(c['tenors'])} tenors, 1Y={c['spots'][0]*100:.4f}%, 20Y={c['spots'][19]*100:.4f}%")

# COMMAND ----------
# Cell 3: Compute Forward Rates & Discount Factors

def compute_forward_rates(tenors, spots):
    """1Y forward rate: f(t→t+1) = ((1+s(t+1))^(t+1) / (1+s(t))^t) - 1"""
    fwd_tenors, fwd_rates = [], []
    for i in range(len(tenors) - 1):
        t1, t2 = tenors[i], tenors[i+1]
        s1, s2 = spots[i], spots[i+1]
        fwd = ((1 + s2)**t2 / (1 + s1)**t1) - 1
        fwd_tenors.append(t2)
        fwd_rates.append(fwd)
    return fwd_tenors, fwd_rates

def compute_discount_factors(tenors, spots):
    """DF(t) = 1 / (1 + s(t))^t"""
    return [1.0 / (1 + s)**t for t, s in zip(tenors, spots)]

# Derive all
for vid in curves:
    c = curves[vid]
    c["fwd_tenors"], c["fwd_rates"] = compute_forward_rates(c["tenors"], c["spots"])
    c["dfs"] = compute_discount_factors(c["tenors"], c["spots"])

# Verify UFR convergence
for vid in curves:
    fwd_60 = curves[vid]["fwd_rates"][58] * 100
    print(f"{vid}: Forward(60Y) = {fwd_60:.4f}%  (UFR target = {UFR*100:.2f}%)")

# COMMAND ----------
# Cell 4: GRAPH 1 — Spot Rate: no_VA vs with_VA (2025-Q4)

fig, ax = plt.subplots(figsize=(14, 6))

for vid in ["RFR_20251231_noVA", "RFR_20251231_withVA"]:
    tenors = curves[vid]["tenors"][:150]
    spots  = [s * 100 for s in curves[vid]["spots"][:150]]
    ls = "-" if "noVA" in vid else "--"
    ax.plot(tenors, spots, color=COLORS[vid], linewidth=2, linestyle=ls, label=LABELS[vid])

# LLP & UFR markers
ax.axvline(x=LLP, color="#555", linestyle=":", linewidth=1.2, alpha=0.7)
ax.axhline(y=UFR * 100, color="#888", linestyle="-.", linewidth=1, alpha=0.5)

ax.annotate("LLP = 20Y", xy=(LLP, ax.get_ylim()[0]),
            xytext=(LLP + 1.5, ax.get_ylim()[0] + 0.15),
            fontsize=9, color="#555", fontweight="bold")
ax.annotate(f"UFR = {UFR*100:.2f}%", xy=(75, UFR*100),
            xytext=(65, UFR*100 + 0.08), fontsize=9, color="#888")

# Zone shading
ylims = ax.get_ylim()
ax.fill_betweenx(ylims, 0, LLP, alpha=0.04, color="green")
ax.fill_betweenx(ylims, LLP, 80, alpha=0.04, color="orange")
ax.text(10, ylims[1] - 0.05, "Market Data", ha="center", fontsize=8, color="green", alpha=0.6)
ax.text(50, ylims[1] - 0.05, "Extrapolation (Smith-Wilson)", ha="center", fontsize=8, color="orange", alpha=0.6)
ax.set_ylim(ylims)

ax.set_xlabel("Maturity (Years)", fontsize=12)
ax.set_ylabel("Spot Rate (%)", fontsize=12)
ax.set_title("EUR Spot Rate Curve: Impact of Volatility Adjustment\nEIOPA RFR 2025-Q4", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.grid(True, alpha=0.2)
ax.set_xlim(0, 150)
fig.tight_layout()
plt.show()

# COMMAND ----------
# Cell 5: GRAPH 2 — Forward Rate: UFR Convergence (2025-Q4)

fig, ax = plt.subplots(figsize=(14, 6))

for vid in ["RFR_20251231_noVA", "RFR_20251231_withVA"]:
    ftenors = curves[vid]["fwd_tenors"]
    frates  = [f * 100 for f in curves[vid]["fwd_rates"]]
    ls = "-" if "noVA" in vid else "--"
    ax.plot(ftenors, frates, color=COLORS[vid], linewidth=2, linestyle=ls, label=LABELS[vid])

ax.axvline(x=LLP, color="#555", linestyle=":", linewidth=1.2, alpha=0.7)
ax.axvline(x=CONV, color="#aa5555", linestyle=":", linewidth=1, alpha=0.5)
ax.axhline(y=UFR * 100, color="#888", linestyle="-.", linewidth=1, alpha=0.5)

ax.annotate("LLP = 20Y", xy=(LLP, 2.8), xytext=(LLP + 1, 2.7),
            fontsize=9, color="#555", fontweight="bold")
ax.annotate("Convergence = 60Y", xy=(CONV, 3.0), xytext=(CONV + 2, 2.85),
            fontsize=9, color="#aa5555")
ax.annotate(f"UFR = {UFR*100:.2f}%", xy=(75, UFR*100),
            xytext=(65, UFR*100 + 0.12), fontsize=9, color="#888")

ax.set_xlabel("Maturity (Years)", fontsize=12)
ax.set_ylabel("1Y Forward Rate (%)", fontsize=12)
ax.set_title("EUR Forward Rate Curve: Convergence to UFR\nEIOPA RFR 2025-Q4", fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.grid(True, alpha=0.2)
ax.set_xlim(0, 150)
fig.tight_layout()
plt.show()

# COMMAND ----------
# Cell 6: GRAPH 3 — 2025-Q4 vs 2026-Q1 (Market Change vs Long-Term Stability)

fig, ax = plt.subplots(figsize=(14, 6))

for vid in ["RFR_20251231_noVA", "RFR_20260331_noVA"]:
    tenors = curves[vid]["tenors"][:150]
    spots  = [s * 100 for s in curves[vid]["spots"][:150]]
    ax.plot(tenors, spots, color=COLORS[vid], linewidth=2, label=LABELS[vid])

ax.axvline(x=LLP, color="#555", linestyle=":", linewidth=1.2, alpha=0.7)
ax.axhline(y=UFR * 100, color="#888", linestyle="-.", linewidth=1, alpha=0.5)

# Highlight market divergence zone
ax.fill_between(
    curves["RFR_20251231_noVA"]["tenors"][:LLP],
    [s * 100 for s in curves["RFR_20251231_noVA"]["spots"][:LLP]],
    [s * 100 for s in curves["RFR_20260331_noVA"]["spots"][:LLP]],
    alpha=0.15, color="#e67e22", label="Market divergence (≤LLP)"
)

ax.annotate("LLP = 20Y", xy=(LLP, 2.5), xytext=(LLP + 1.5, 2.45),
            fontsize=9, color="#555", fontweight="bold")

ax.set_xlabel("Maturity (Years)", fontsize=12)
ax.set_ylabel("Spot Rate (%)", fontsize=12)
ax.set_title("EUR Spot Rate: Market Changes vs Long-Term Stability\n2025-Q4 vs 2026-Q1 (no VA)", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.grid(True, alpha=0.2)
ax.set_xlim(0, 150)
fig.tight_layout()
plt.show()

# COMMAND ----------
# Cell 7: GRAPH 4 — Discount Factor Curve (2025-Q4)

fig, ax = plt.subplots(figsize=(14, 6))

for vid in ["RFR_20251231_noVA", "RFR_20251231_withVA"]:
    tenors = curves[vid]["tenors"][:150]
    dfs    = curves[vid]["dfs"][:150]
    ls = "-" if "noVA" in vid else "--"
    ax.plot(tenors, dfs, color=COLORS[vid], linewidth=2, linestyle=ls, label=LABELS[vid])

ax.axvline(x=LLP, color="#555", linestyle=":", linewidth=1.2, alpha=0.7)

# Annotate 20Y DF values
df_20_noVA = curves["RFR_20251231_noVA"]["dfs"][19]
df_20_wVA  = curves["RFR_20251231_withVA"]["dfs"][19]

ax.annotate(f"DF(20Y) no VA = {df_20_noVA:.4f}",
            xy=(20, df_20_noVA), xytext=(28, df_20_noVA + 0.04),
            fontsize=9, color=COLORS["RFR_20251231_noVA"],
            arrowprops=dict(arrowstyle="->", color=COLORS["RFR_20251231_noVA"], lw=0.8))
ax.annotate(f"DF(20Y) with VA = {df_20_wVA:.4f}",
            xy=(20, df_20_wVA), xytext=(28, df_20_wVA - 0.06),
            fontsize=9, color=COLORS["RFR_20251231_withVA"],
            arrowprops=dict(arrowstyle="->", color=COLORS["RFR_20251231_withVA"], lw=0.8))

diff_pct = (df_20_wVA - df_20_noVA) / df_20_noVA * 100
ax.text(42, 0.55, f"VA effect at 20Y:\nDF difference = {diff_pct:.2f}%",
        fontsize=10, bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.8))

ax.set_xlabel("Maturity (Years)", fontsize=12)
ax.set_ylabel("Discount Factor", fontsize=12)
ax.set_title("EUR Discount Factor Curve: Time Value of Money\nEIOPA RFR 2025-Q4", fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=10)
ax.grid(True, alpha=0.2)
ax.set_xlim(0, 150)
ax.set_ylim(0, 1.05)
fig.tight_layout()
plt.show()

# COMMAND ----------
# Cell 8: GRAPH 5 — BEL Impact by Curve Version

# Load BEL results from mart
df_bel = spark.sql("""
    select scenario_id, version_id, total_bel
    from workspace.life_insurance_bel_mart.mart_portfolio_overview
    where scenario_id = 'BASE'
    order by version_id
""").toPandas()

fig, ax = plt.subplots(figsize=(12, 7))

versions = df_bel["version_id"].tolist()
bels = (df_bel["total_bel"] / 1e6).tolist()
colors = [COLORS.get(v, "#333") for v in versions]
labels = [LABELS.get(v, v) for v in versions]

bars = ax.bar(range(len(versions)), bels, color=colors, width=0.6, edgecolor="white", linewidth=1.5)

# Value labels on bars
for i, (bar, bel) in enumerate(zip(bars, bels)):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{bel:.2f}M", ha="center", va="bottom", fontsize=11, fontweight="bold",
            color=colors[i])

ax.set_xticks(range(len(versions)))
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("Total BEL (€ millions)", fontsize=12)
ax.set_title("BEL Impact: Same Portfolio, Different Discount Curves\nBASE Scenario", fontsize=14, fontweight="bold")

# Delta annotations
max_bel = max(bels)
min_bel = min(bels)
delta = max_bel - min_bel
delta_pct = delta / max_bel * 100
ax.text(0.98, 0.95,
        f"Max difference: €{delta:.2f}M ({delta_pct:.1f}%)",
        transform=ax.transAxes, ha="right", va="top", fontsize=11,
        bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.8))

ax.grid(True, alpha=0.2, axis="y")
ax.set_ylim(min(bels) * 0.97, max(bels) * 1.04)
fig.tight_layout()
plt.show()

# COMMAND ----------
# Cell 9: Summary Table

print("=" * 70)
print("EIOPA RFR Curve Comparison — EUR BASE Scenario")
print("=" * 70)

for _, row in df_bel.iterrows():
    vid = row["version_id"]
    bel = row["total_bel"]
    print(f"  {LABELS.get(vid, vid):25s}  BEL = €{bel/1e6:,.2f}M")

print("-" * 70)
ref_bel = df_bel[df_bel["version_id"] == "RFR_20251231_noVA"]["total_bel"].values[0]
for _, row in df_bel.iterrows():
    vid = row["version_id"]
    bel = row["total_bel"]
    delta = bel - ref_bel
    pct = delta / abs(ref_bel) * 100
    print(f"  {LABELS.get(vid, vid):25s}  Δ vs 2025-Q4 noVA = €{delta/1e6:+,.2f}M ({pct:+.2f}%)")
print("=" * 70)